# CS 228 pipeline: one hands-off run (Kaggle)

**Use:** Session options -> Accelerator = GPU T4 x2, Internet = On. Then **Save Version -> Save & Run All (Commit)**.
It runs top to bottom on its own (about 2 hours), so you can close the tab. Results: Versions -> open the newest -> Output
(`bundle2.zip` and the log). Do NOT run it cell by cell and leave the tab idle: interactive sessions time out.

In [ ]:
# 1. Get the code
%cd /kaggle/working
!rm -rf cs228-pad-project
GITHUB_USER = "ShivamShelar101"
!git clone https://github.com/{GITHUB_USER}/cs228-pad-project.git
%cd cs228-pad-project
!pip install -q onnxscript matplotlib
!nvidia-smi -L

In [ ]:
# 2. Dataset location
import os
SRC = "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing"
if not os.path.exists(SRC):
    import kagglehub
    SRC = kagglehub.dataset_download("attentionlayer241/celeba-spoof-for-face-antispoofing")
print("dataset at:", SRC)

In [ ]:
# 3. Face crops: many people, few photos each (about 30 minutes)
!python -m data.prepare_celeba_spoof --src {SRC} --dst ./data_pad2 --per_class 3000 --test_per_class 600 --max_per_subject 8
!for s in train val test; do for c in live spoof; do echo $s/$c $(ls data_pad2/$s/$c | wc -l); done; done

In [ ]:
# 4. Baseline (no defense)
!python -m training.train_baseline --data_root ./data_pad2 --epochs 5 --seed 0 --out checkpoints/baseline2

In [ ]:
# 5. Hardened model (full method), lower learning rate, 10 epochs
!python -m training.train_hardened --data_root ./data_pad2 --epochs 10 --lr 3e-5 --seed 0 --out checkpoints/full2

In [ ]:
# 6. Evaluate both on the same data + report
!python -m evaluation.evaluate --data_root ./data_pad2 --baseline_glob "checkpoints/baseline2/seed*_best.pt" --hardened_glob "checkpoints/full2/seed*_best.pt" --out results/eval_full2.json
!python -m evaluation.make_report --run "Full method (lr 3e-5, 10 epochs)=results/eval_full2.json" --out results/report2
print(open("results/report2.md").read())

In [ ]:
# 7. Which kinds of fakes does each model accept? (baseline, then hardened)
!python -m evaluation.by_spoof_type --src {SRC} --data_root ./data_pad2 --ckpt checkpoints/baseline2/seed0_best.pt
!python -m evaluation.by_spoof_type --src {SRC} --data_root ./data_pad2 --ckpt checkpoints/full2/seed0_best.pt

In [ ]:
# 8. Pack the results (checkpoints are small; data_pad2 stays in the output for reuse)
!zip -qr bundle2.zip results checkpoints/baseline2/seed0_best.pt checkpoints/full2/seed0_best.pt
!ls -lh bundle2.zip